# LACAN: Molecule Generation

This notebook covers:
1. Random molecule generation (with optional corpus biasing toward known actives)
2. The adaptive GA with explore/exploit logic
3. Shape-based optimization (slow scorer example)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from lacan import lacan
from lacan import gen, replace
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.drawOptions.drawMolsSameScale = False

p = lacan.load_profile("chembl")
print("Profile loaded.")

## 1. Random molecule generation

In [ ]:
ms = gen.generate_filtered_molecules(
    n_jobs=-1, n_molecules=18, profile=p, seed=456, min_atoms=20
)
d = Draw.MolsToGridImage(ms[:18], molsPerRow=6)
display(d)

## 2. Corpus biasing (transfer learning)

`bias_corpus` takes a set of reference molecules (e.g. known actives) and boosts the sampling weight of fragments that appear in them. The `ratio` parameter controls how strongly: `ratio=2` (default) doubles the effective frequency of custom fragments relative to the background ChEMBL corpus.

This steers generation toward similar chemotypes without hard-coding anything.

In [ ]:
# Load known D3 binders as reference chemistry
actives = [m for m in Chem.SmilesMolSupplier("../data/d3_actives.csv") if m]
print(f"Loaded {len(actives)} actives")

# Build biased corpus — fragments from actives get 3x weight
biased_corpus = gen.bias_corpus(actives, ratio=3.0)
print(f"Corpus size: {len(biased_corpus)} fragments")

# Generate molecules using biased corpus
ms_biased = gen.generate_filtered_molecules(
    n_jobs=-1, n_molecules=9, profile=p, seed=456, min_atoms=20,
    fragcorpus=biased_corpus
)
d = Draw.MolsToGridImage(ms_biased, molsPerRow=3)
display(d)

## 3. Scaffold operations (one-shot)

These can be used standalone or are called internally by the GA.

In [ ]:
# Ring replacement on sildenafil
sild = Chem.MolFromSmiles("CCCc1nn(C)c2c(=O)[nH]c(-c3cc(S(=O)(=O)N4CCN(C)CC4)ccc3OCC)nc12")
ms = replace.replace_ring(sild, p, 0.002, n_replacements=100)
d = Draw.MolsToGridImage(ms, molsPerRow=3)
display(d)

## 4. Adaptive GA — fast scoring (QSAR)

The GA adapts each generation based on:
- **diversity**: mean pairwise Tanimoto distance of the pool. Low → explore
- **plateau**: gens without improvement. Stalled → explore

For fast scorers use larger `startN`/`popsize`. `explore_ratio` sets how much of a free-choice generation goes toward coarse fragment operations vs mutations.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from rdkit.Chem import rdFingerprintGenerator
MFPGEN = rdFingerprintGenerator.GetMorganGenerator(2, fpSize=1024)

actives = [m for m in Chem.SmilesMolSupplier("../data/d3_actives.csv") if m]
decoys  = [m for m in Chem.SmilesMolSupplier("../data/d3_decoys.csv") if m]
X = [MFPGEN.GetFingerprintAsNumPy(m) for m in actives + decoys]
y = [1] * len(actives) + [0] * len(decoys)
rfc = RandomForestClassifier(n_jobs=4, max_depth=10, n_estimators=100)
rfc.fit(X, y)

def rfscores(mols):
    fps = [MFPGEN.GetFingerprintAsNumPy(m) for m in mols]
    return [k[1] for k in rfc.predict_proba(fps)]

print("Model trained.")

In [ ]:
winners = gen.generate_optimized_molecules(
    rfscores, p,
    win_threshold=0.6,
    startN=40,
    popsize=30,
    generations=10,
    higher_is_better=True,
    diversity_threshold=0.4,
    plateau_patience=2,
    explore_ratio=0.5,
    quiet=False,
    seed=345,
)
allc = sorted(winners, key=lambda x: -x[1])
print(f"\n{len(allc)} winners found")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc[:18]],
    useSVG=True, molsPerRow=6,
    legends=[str(round(c[1], 3)) for c in allc[:18]],
)
display(d)

## 5. GA reporter — score per generation

Pass a `GAReporter` as `callback=` to capture per-generation stats without changing anything else. After the run, call `.plot()` for a single-run dashboard or `GAReporter.compare([r1, r2, ...])` to overlay multiple runs for settings comparison.

In [ ]:
from lacan.gen import GAReporter

reporter = GAReporter(label="default settings")

winners_r = gen.generate_optimized_molecules(
    rfscores, p,
    win_threshold=0.6,
    startN=40,
    popsize=30,
    generations=10,
    higher_is_better=True,
    diversity_threshold=0.4,
    plateau_patience=2,
    explore_ratio=0.5,
    n_jobs=-1,           # use all CPU cores
    quiet=True,
    seed=345,
    callback=reporter,
)

reporter.plot()
reporter.summary()

### Compare settings side by side

Run the GA with two different `explore_ratio` values and overlay their score curves. Any parameter can be varied this way.

In [ ]:
r_low  = GAReporter(label="explore_ratio=0.3")
r_high = GAReporter(label="explore_ratio=0.8")

base_kwargs = dict(
    scoring_function=rfscores, profile=p,
    win_threshold=0.6, startN=40, popsize=30,
    generations=10, higher_is_better=True,
    diversity_threshold=0.4, plateau_patience=2,
    n_jobs=-1, quiet=True, seed=345,
)

gen.generate_optimized_molecules(**base_kwargs, explore_ratio=0.3, callback=r_low)
gen.generate_optimized_molecules(**base_kwargs, explore_ratio=0.8, callback=r_high)

GAReporter.compare([r_low, r_high], metric="best_pool")

### GA + biased corpus

Combine the GA with corpus biasing to steer toward a target chemotype while still optimizing the objective.

In [ ]:
winners_biased = gen.generate_optimized_molecules(
    rfscores, p,
    win_threshold=0.6,
    startN=40,
    popsize=30,
    generations=10,
    higher_is_better=True,
    diversity_threshold=0.4,
    plateau_patience=2,
    explore_ratio=0.5,
    quiet=True,
    seed=345,
    fragcorpus=biased_corpus,   # <-- bias toward D3 actives chemistry
)
allc_b = sorted(winners_biased, key=lambda x: -x[1])
print(f"{len(allc_b)} winners found with biased corpus")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_b[:18]],
    useSVG=True, molsPerRow=6,
    legends=[str(round(c[1], 3)) for c in allc_b[:18]],
)
display(d)

## 6. Adaptive GA — slow scoring (3D shape alignment)

For slow scorers, use small `startN`/`popsize` and higher `explore_ratio` to make big jumps efficiently. The GA's safe scoring wrapper catches any 3D embedding failures and assigns score 0 rather than crashing.

We use **vortioxetine** as the reference compound for shape similarity optimization.

In [ ]:
# Vortioxetine as reference (loaded from the uploaded SDF)
from rdkit.Chem import SDMolSupplier, rdDistGeom, rdShapeAlign, AddHs

refmol = next(m for m in SDMolSupplier("vortioxetine.sdf") if m is not None)
print(f"Reference: {Chem.MolToSmiles(refmol)}")

def get_alignment_scores(mols, ref=refmol):
    scores = []
    for m in mols:
        try:
            m = AddHs(m)
            rdDistGeom.EmbedMultipleConfs(m, numConfs=20, randomSeed=0xf00d)
            conf_scores = []
            for confId in range(m.GetNumConformers()):
                conf_scores.append(
                    sum(rdShapeAlign.AlignMol(ref, m, probeConfId=confId)) / 2
                )
            scores.append(max(conf_scores) if conf_scores else 0.0)
        except Exception:
            scores.append(0.0)
    return scores

In [ ]:
winners_shape = gen.generate_optimized_molecules(
    get_alignment_scores, p,
    startN=6,
    popsize=4,
    win_threshold=0.7,
    generations=10,
    higher_is_better=True,
    diversity_threshold=0.5,
    plateau_patience=2,
    explore_ratio=0.7,
    n_jobs=1,            # slow scorer: don't compete with its own parallelism
    quiet=False,
    seed=888,
)
allc_shape = sorted(winners_shape, key=lambda x: -x[1])
print(f"\n{len(allc_shape)} winners found")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_shape],
    useSVG=True, molsPerRow=4,
    legends=[str(round(c[1], 3)) for c in allc_shape],
    maxMols=20,
)
display(d)